# Student Productivity vs. Social Media Usage
## Kujenga Data Science Final Project

**The Big Question:** As students, we all feel the pull of social media. but does spending more time scrolling actually hurt our productivity? Let's look at the data to find out.

---


## 1. Introduction: Why are we doing this?

We’ve all been there: you open an app for 'just five minutes' and suddenly an hour has passed. But beyond just losing time, how does this habit affect our actual ability to get things done?

In this project, we aren't just guessing. We are using data from thousands of students to see if there is a real, measurable link between the hours we spend on social media and our productivity levels. Our goal is to see if we can predict how much productivity we 'lose' for every extra hour of screen time.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pylab import rcParams
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from statsmodels.formula.api import ols

import warnings
warnings.filterwarnings('ignore')

# Making the plots look clean and professional
rcParams['figure.figsize'] = 14/2.54, 14/2.54
matplotlib.font_manager.FontProperties(family='Helvetica',size=11)
sns.set_style('whitegrid')
print("Tools ready!")

## 2. Getting the Data Ready

Real-world data is usually a bit messy—it has missing pieces, duplicates, and weird formatting. Before we can trust our results, we need to clean it up. We’ll follow four simple steps:

### Step 1: Loading and Taking a First Look

First, we load the full dataset of 30,000 people and filter it so we are only looking at **Students**. After all, we want to know how this affects *us*.

In [ ]:
# Load the data
df_full = pd.read_csv('social_media_vs_productivity.csv')

# Keep only the students
df = df_full[df_full['job_type'] == 'Student'].copy()

print(f"We have {len(df)} student records to work with. Here's a quick look at the first few:")
display(df.head())


### Step 2: Handling Missing Pieces

Sometimes students forget to fill out every part of a survey. For things like 'stress levels,' we fill in the blanks with the 'middle' (median) value. However, if a record is missing the most important info—like how much time they spent online or their productivity score—we have to remove it because we shouldn't guess those.

In [ ]:
# Fill in missing scores with the median
for col in ['stress_level', 'sleep_hours', 'job_satisfaction_score']:
    df[col].fillna(df[col].median(), inplace=True)

# Remove rows where our main data is missing
df.dropna(subset=['daily_social_media_time', 'actual_productivity_score'], inplace=True)
print("Missing values handled!")


### Step 3: Removing Double-Entries

If a student's data was accidentally entered twice, it could skew our results. We make sure every student in our list is unique.

In [ ]:
df.drop_duplicates(inplace=True)
print(f"Clean records remaining: {len(df)}")


### Step 4: Making Sure the Numbers are 'Numbers'

We check to make sure Python sees our scores as numbers it can do math with, not just pieces of text.

In [ ]:
df['daily_social_media_time'] = pd.to_numeric(df['daily_social_media_time'], errors='coerce')
df['actual_productivity_score'] = pd.to_numeric(df['actual_productivity_score'], errors='coerce')

# Convert platform to a category
df['social_platform_preference'] = df['social_platform_preference'].astype('category')

print("Data types verified.")


## 3. Visualizing the Relationship

Let's put the data on a map. Each gray dot below is a student. We've highlighted a few random students in black so you can see where they stand.

**What to look for:** Does the cloud of dots generally go 'downhill' as we move to the right (more hours)?


In [ ]:
def plotData(df, x_col, y_col): 
    fig, ax = plt.subplots(num=1)
    ax.plot(x_col, y_col, data=df, linestyle='none', markersize=4, marker='o', color=[0.85, 0.85, 0.85], alpha=0.5)
    
    # Highlight some examples to make it more human
    highlights = df.sample(5, random_state=42).index
    for idx in highlights:
        ax.plot(df.loc[idx, x_col], df.loc[idx, y_col], linestyle='none', markersize=6, marker='o', color='black')
        ax.text(df.loc[idx, x_col]+0.2, df.loc[idx, y_col]+0.1, f"Student {idx}", fontsize=9)
           
    ax.set_ylabel('Productivity Score (0-10)')
    ax.set_xlabel('Daily Social Media Time (Hours)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    return fig, ax

fig, ax = plotData(df, 'daily_social_media_time', 'actual_productivity_score')
ax.set_title("How Social Media Time Affects Student Productivity")
plt.show()


## 4. The 'Price' of an Hour (Mathematical Modeling)

Now, we want to find the 'Best Fit Line.' This is a straight line that passes through the middle of all those dots. 

The 'Slope' of this line is the most important part: it tells us exactly how much productivity goes down for every extra hour a student spends on social media.


In [ ]:
# Doing the math manually (following the 'How to be Happy' lesson)
# 1. Calculate the means
x_mean = np.mean(df['daily_social_media_time'])
y_mean = np.mean(df['actual_productivity_score'])

# 2. Shift the data (Centering)
df['x_shifted'] = df['daily_social_media_time'] - x_mean
df['y_shifted'] = df['actual_productivity_score'] - y_mean

# 3. Calculate the best slope (m)
m_best = np.sum(df['x_shifted'] * df['y_shifted']) / np.sum(df['x_shifted']**2)

# 4. Calculate the intercept (k)
k_best = y_mean - m_best * x_mean

print(f"The Manual Calculation:")
print(f"Slope (m) = {m_best:.4f}")
print(f"Intercept (k) = {k_best:.4f}")


### Proving it's the 'Best Fit': Sum of Squares

How do we know this line is the 'best'? We check the **Sum of Squares**. We calculate the distance between every student's real score and what the line predicts. The line that makes these squared distances the smallest is the 'Winner.'

In [ ]:
# Calculate predictions for every student
df['Predicted_Productivity'] = m_best * df['daily_social_media_time'] + k_best

# Calculate squared distances (Errors)
df['Squared_Error'] = (df['actual_productivity_score'] - df['Predicted_Productivity'])**2

# Sum them up
Total_Sum_Of_Squares = np.sum(df['Squared_Error'])

print(f"The Total Sum of Squares for the best fit line is: {Total_Sum_Of_Squares:.4f}")
print("This is the minimum possible error for a straight line through this data!")


### Visualizing the Best Fit Line

Here is the calculated line drawn over the data. This line shows the average 'path' of productivity as screen time increases. You can see how clearly it points downward!

In [ ]:
# Generate points for the best fit line
x_range = np.linspace(0, df['daily_social_media_time'].max(), 100)
y_range = m_best * x_range + k_best

# Plot the data again
fig, ax = plotData(df, 'daily_social_media_time', 'actual_productivity_score')

# Add the line
ax.plot(x_range, y_range, linestyle='-', color='black', linewidth=2)
ax.set_title("The 'Best Fit' Line Showing the Downward Trend")
plt.show()


## 5. Interactive Dashboard (Streamlit)

To make this analysis even more accessible and human, we have created an interactive **Streamlit Dashboard**. 

This app allows you to:
- Filter the data by specific social media platforms.
- Use an interactive slider to **predict your own productivity score** based on our model.
- Explore the relationship between stress and digital habits in real-time.

### How to run the Dashboard:
1. Open your terminal in this project folder.
2. Run the following command:
   ```bash
   streamlit run app.py
   ```


## 6. What did we learn? (Final Conclusion)

After looking at the data from over 5,000 students, we can draw some pretty clear conclusions—but we also have to be careful with how we interpret them.

### 1. The Numbers Don't Lie
Based on this specific data, there is a clear link: **more social media time = lower productivity.** 
- The model shows a 'Pearson Correlation' of about **-0.6**. In simple terms, this means that as one goes up, the other almost always goes down.
- For every **extra hour** you spend scrolling, your productivity score drops by about **0.45 points** (on a 10-point scale).

### 2. A More Human Perspective (The Stress-Relief Hypothesis)
While the numbers show a drop, we have to ask *why*. Is social media *causing* the low productivity? Or is it something else?
- **The Coping Mechanism:** It’s possible that when we feel stressed or unproductive, we turn to social media to 'escape' or 'relax.' 
- This means that for some students, high social media use might be a **symptom** of stress, not the original cause.

### 3. Final Takeaway
In conclusion, while social media usage is statistically linked to decreased productivity in this study, the underlying reasons are likely a complex mix of distraction and stress response. Keeping an eye on your screen time is a great first step toward getting more done!

### For more information and clearer results view the streamlit application : https://socialmediavsappuctivityapp.streamlit.app/
